# Notebook 05 — Gold Layer: Janelas Temporais de Demanda
## Projeto: Otimização de Frotas Citi Bike NYC
### Integrante 4 — PySpark Pipeline Lead

---

**Objetivo:** Transformar os dados limpos da camada **Silver/trips** (Delta Lake, Integrante 3)
em agregações de demanda por estação com janelas temporais de **15 e 30 minutos**,
gerando alertas de rebalanceamento e perfis históricos para o modelo preditivo.

**Dependência:** Execute o **Notebook 03** (Silver Layer) antes deste.

**Pipeline:**
```
Silver/trips (Delta Lake — Int. 3)
    │
    ├── Enriquecimento temporal (janelas 15/30 min, flag de pico)
    │
    ├── demand_15min          → Gold/Delta  (Int. 5 e Int. 6)
    ├── demand_30min          → Gold/Delta  (Int. 5 e Int. 6)
    ├── demand_rolling_avg    → Gold/Delta  (Int. 5 — modelo preditivo)
    └── station_hourly_profile→ Gold/Delta  (Int. 5 — feature engineering)
```

**Tabelas Gold produzidas:**
| Tabela | Grão | Uso |
|---|---|---|
| `demand_15min` | estação × janela 15 min | Demanda em alta resolução para alertas |
| `demand_30min` | estação × janela 30 min | Demanda suavizada para previsão |
| `demand_rolling_avg` | estação × janela 15 min | Média móvel (1h) + `rebalance_alert` |
| `station_hourly_profile` | estação × hora × dia da semana | Feature histórica para ML |

## 0. Setup & Dependências

In [1]:
# delta-spark já instalado no Dockerfile do projeto
# Se rodar fora do Docker, descomente:
# !pip install pyspark==3.5.2 delta-spark==3.2.0 pyarrow pandas -q

In [2]:
import os
import sys
import logging
from pathlib import Path

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import BooleanType
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger('citibike_int4')

print(f'Python {sys.version}')
print(f'Pandas {pd.__version__}')

Python 3.11.6 | packaged by conda-forge | (main, Oct  3 2023, 10:40:35) [GCC 12.3.0]
Pandas 2.0.3


In [3]:
# ============================================================
# CONFIGURAÇÃO — mesmo padrão dos notebooks 01-04
# ============================================================

# No Docker: cwd = /home/jovyan/work/notebooks/
# PROJECT_ROOT = /home/jovyan/work/
PROJECT_ROOT = Path(os.getcwd()).parent
DATA_DIR     = PROJECT_ROOT / 'dados'

# Entrada — Silver 
SILVER_TRIPS = DATA_DIR / 'silver' / 'trips'

# Saídas — Gold 
GOLD_DEMAND_15MIN          = DATA_DIR / 'gold' / 'demand_15min'
GOLD_DEMAND_30MIN          = DATA_DIR / 'gold' / 'demand_30min'
GOLD_DEMAND_ROLLING        = DATA_DIR / 'gold' / 'demand_rolling_avg'
GOLD_STATION_PROFILE       = DATA_DIR / 'gold' / 'station_hourly_profile'

# Criar pastas de saída (caso não existam)
for d in [GOLD_DEMAND_15MIN, GOLD_DEMAND_30MIN,
          GOLD_DEMAND_ROLLING, GOLD_STATION_PROFILE]:
    d.mkdir(parents=True, exist_ok=True)

# Verificar se Silver existe
if not any(SILVER_TRIPS.rglob('*.parquet')):
    raise FileNotFoundError(
        f'Silver/trips vazio: {SILVER_TRIPS}\n'
        'Execute o Notebook 03 (Silver Layer) antes deste.'
    )

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'Silver/trips : {SILVER_TRIPS}  ✅')
print(f'Gold output  : {DATA_DIR / "gold"}')

PROJECT_ROOT : /home/jovyan/work
Silver/trips : /home/jovyan/work/dados/silver/trips  ✅
Gold output  : /home/jovyan/work/dados/gold


In [4]:
# ============================================================
# SparkSession — mesma config do Dockerfile
# (Delta Lake 3.2.0 + Spark 3.5.2)
# ============================================================

# Parar sessão anterior se existir
_active = SparkSession.getActiveSession()
if _active:
    _active.stop()

spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .appName('CitiBike_Int4_Pipeline')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.session.timeZone', 'America/New_York')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.databricks.delta.retentionDurationCheck.enabled', 'false')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .master('local[*]')
).getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} + Delta Lake inicializado')
print(f'  Cores: {spark.sparkContext.defaultParallelism}')
print(f'  Driver memory: {spark.conf.get("spark.driver.memory")}')

Spark 3.5.0 + Delta Lake inicializado
  Cores: 12
  Driver memory: 8g


---
## 1. Leitura da Silver/trips

Consumindo o Delta Lake produzido pelo **Integrante 3** (Notebook 03).
Particionado por `(year, month)`, schema completo com features de clima,
feriados, zona NYC e distância Haversine já calculadas.

In [5]:
# ============================================================
# 1.1 Carregar Silver/trips (Delta Lake)
# ============================================================

silver_path_str = str(SILVER_TRIPS).replace('\\', '/')

try:
    df_silver = spark.read.format('delta').load(silver_path_str)
    print('Formato: Delta Lake')
except Exception as e:
    print(f'Delta falhou ({e}), tentando Parquet...')
    df_silver = spark.read.parquet(silver_path_str)
    print('Formato: Parquet (fallback)')

total = df_silver.count()
print(f'\nTotal de registros Silver: {total:,}')
print(f'Colunas: {len(df_silver.columns)}')
df_silver.printSchema()

Formato: Delta Lake

Total de registros Silver: 499,307
Colunas: 36
root
 |-- ride_id: string (nullable = true)
 |-- rideable_type: string (nullable = true)
 |-- member_casual: string (nullable = true)
 |-- started_at: timestamp (nullable = true)
 |-- ended_at: timestamp (nullable = true)
 |-- trip_duration_sec: long (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- date: date (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- day_name: string (nullable = true)
 |-- is_weekend: boolean (nullable = true)
 |-- season: string (nullable = true)
 |-- is_holiday: boolean (nullable = true)
 |-- holiday_name: string (nullable = true)
 |-- weather_temp_c: double (nullable = true)
 |-- weather_precip_mm: double (nullable = true)
 |-- weather_is_raining: boolean (nullable = true)
 |-- weather_code: long 

In [6]:
# ============================================================
# 1.2 Amostra dos dados de entrada
# ============================================================

df_silver.select(
    'ride_id', 'started_at', 'ended_at',
    'start_station_id', 'start_station_name',
    'trip_duration_sec', 'member_casual'
).show(5, truncate=False)

+----------------+-----------------------+-----------------------+----------------+---------------------+-----------------+-------------+
|ride_id         |started_at             |ended_at               |start_station_id|start_station_name   |trip_duration_sec|member_casual|
+----------------+-----------------------+-----------------------+----------------+---------------------+-----------------+-------------+
|000051824797E5E9|2026-01-09 11:36:40.672|2026-01-09 11:52:21.64 |6907.03         |46 St & 25 Ave       |941              |member       |
|000077C2CED6D085|2026-01-10 17:42:56.481|2026-01-10 17:57:09.765|6332.10         |Broadway & 74 St     |853              |member       |
|0000917896A97F20|2026-01-08 10:53:11.152|2026-01-08 10:58:41.106|4060.09         |Carroll St & 5 Ave   |330              |member       |
|000091876686F2B7|2026-01-13 09:14:51.045|2026-01-13 09:19:42.223|7286.01         |E 91 St & 2 Ave      |291              |casual       |
|0000B9ED16E2ACA9|2026-01-12 12:44

---
## 2. Enriquecimento Temporal

Adição das colunas de janela temporal (15 e 30 min) e flags de horário de pico.

**Lógica de truncamento (sem UDF — 100% colunar):**
```
window_15min = date_trunc('minute', started_at) - (minute(started_at) % 15) MINUTES
window_30min = date_trunc('minute', started_at) - (minute(started_at) % 30) MINUTES
```

**Exemplo:**
| started_at | window_15min | window_30min |
|---|---|---|
| 08:17:33 | 08:15:00 | 08:00:00 |
| 08:31:45 | 08:30:00 | 08:30:00 |
| 17:48:12 | 17:45:00 | 17:30:00 |

In [7]:
df_enriched = (
    df_silver
    .withColumn('hour_of_day', F.hour('started_at'))
    .withColumn('day_of_week', F.dayofweek('started_at'))
    .withColumn('_ts', F.unix_timestamp('started_at'))
    .withColumn('_min', F.minute('started_at'))
    # Janelas como STRING (evita problema de interval type no Spark 3.5)
    .withColumn('window_15min',
        F.from_unixtime(F.col('_ts') - (F.col('_min') % 15) * 60))
    .withColumn('window_30min',
        F.from_unixtime(F.col('_ts') - (F.col('_min') % 30) * 60))
    .drop('_ts', '_min')
    .withColumn('is_peak_hour',
        F.when(
            F.col('hour_of_day').between(7, 9) |
            F.col('hour_of_day').between(17, 19), True
        ).otherwise(False).cast(BooleanType()))
)

In [8]:
# ============================================================
# 2.2 Verificar truncamento das janelas (amostra)
# ============================================================

df_enriched.select(
    'started_at', 'window_15min', 'window_30min',
    'hour_of_day', 'is_peak_hour'
).show(10, truncate=False)

+-----------------------+-------------------+-------------------+-----------+------------+
|started_at             |window_15min       |window_30min       |hour_of_day|is_peak_hour|
+-----------------------+-------------------+-------------------+-----------+------------+
|2026-01-09 11:36:40.672|2026-01-09 11:30:40|2026-01-09 11:30:40|11         |false       |
|2026-01-10 17:42:56.481|2026-01-10 17:30:56|2026-01-10 17:30:56|17         |true        |
|2026-01-08 10:53:11.152|2026-01-08 10:45:11|2026-01-08 10:30:11|10         |false       |
|2026-01-13 09:14:51.045|2026-01-13 09:00:51|2026-01-13 09:00:51|9          |true        |
|2026-01-12 12:44:32.918|2026-01-12 12:30:32|2026-01-12 12:30:32|12         |false       |
|2026-01-05 13:38:30.338|2026-01-05 13:30:30|2026-01-05 13:30:30|13         |false       |
|2026-01-04 10:55:39.043|2026-01-04 10:45:39|2026-01-04 10:30:39|10         |false       |
|2026-01-06 04:05:09.836|2026-01-06 04:00:09|2026-01-06 04:00:09|4          |false       |

In [9]:
# ============================================================
# 2.3 Distribuição de viagens por hora (sanidade)
# ============================================================

print('Distribuição por hora do dia:')
(
    df_enriched
    .groupBy('hour_of_day', 'is_peak_hour')
    .count()
    .orderBy('hour_of_day')
    .show(24, truncate=False)
)

Distribuição por hora do dia:
+-----------+------------+-----+
|hour_of_day|is_peak_hour|count|
+-----------+------------+-----+
|0          |false       |4656 |
|1          |false       |12647|
|2          |false       |26373|
|3          |false       |38980|
|4          |false       |29636|
|5          |false       |23811|
|6          |false       |24647|
|7          |true        |27818|
|8          |true        |27977|
|9          |true        |30181|
|10         |false       |32942|
|11         |false       |37455|
|12         |false       |45683|
|13         |false       |40769|
|14         |false       |26340|
|15         |false       |18228|
|16         |false       |14027|
|17         |true        |11252|
|18         |true        |8462 |
|19         |true        |6254 |
|20         |false       |4187 |
|21         |false       |3022 |
|22         |false       |1992 |
|23         |false       |1968 |
+-----------+------------+-----+



---
## 3. Gold: `demand_15min`

**Grão:** `(station_id, window_15min)`

Agrega partidas e chegadas por estação a cada 15 minutos.
`net_flow = arrivals - departures`
- Positivo → estação acumulando bikes
- Negativo → estação perdendo bikes (risco de esvaziamento)

In [10]:
# Aumentar partições para aliviar memória nas agregações
spark.conf.set('spark.sql.shuffle.partitions', '64')

# Liberar cache
df_enriched.unpersist()

# Reler direto da Silver sem cache
df_enriched = spark.read.format('delta').load(str(SILVER_TRIPS).replace('\\', '/'))

# Adicionar colunas de janela sem cache
df_enriched = (
    df_enriched
    .withColumn('_min', F.minute('started_at'))
    .withColumn('window_15min',
        F.date_trunc('minute', F.col('started_at'))
        - (F.col('_min') % 15 * 60).cast('long').cast('timestamp'))
    .withColumn('window_30min',
        F.date_trunc('minute', F.col('started_at'))
        - (F.col('_min') % 30 * 60).cast('long').cast('timestamp'))
    .drop('_min')
    .withColumn('hour_of_day', F.hour('started_at'))
    .withColumn('day_of_week', F.dayofweek('started_at'))
    .withColumn('is_peak_hour',
        F.when(
            F.col('hour_of_day').between(7, 9) |
            F.col('hour_of_day').between(17, 19), True
        ).otherwise(False).cast(BooleanType()))
)

print('Pronto — sem cache, 64 partições')

Pronto — sem cache, 64 partições


In [11]:
# ============================================================
# 3.1 Partidas por estação × janela 15min
# ============================================================

df_dep_15 = (
    df_enriched
    .groupBy(
        F.col('start_station_id').alias('station_id'),
        F.col('start_station_name').alias('station_name'),
        'window_15min',
        'hour_of_day',
        'day_of_week',
        'is_peak_hour',
    )
    .agg(
        F.count('ride_id').alias('departures'),
        F.sum(
            F.when(F.col('is_peak_hour'), 1).otherwise(0)
        ).alias('peak_departures'),
        F.round(
            F.avg('trip_duration_sec') / 60.0, 2
        ).alias('avg_trip_duration_min'),
    )
)

# ============================================================
# 3.2 Chegadas por estação × janela 15min
# ============================================================

df_arr_15 = (
    df_enriched
    .groupBy(
        F.col('end_station_id').alias('station_id'),
        'window_15min',
    )
    .agg(
        F.count('ride_id').alias('arrivals'),
        F.sum(
            F.when(F.col('is_peak_hour'), 1).otherwise(0)
        ).alias('peak_arrivals'),
    )
)

# ============================================================
# 3.3 Join e net_flow
# ============================================================
df_demand_15min = (
    df_dep_15
    .join(df_arr_15, on=['station_id', 'window_15min'], how='left')
    .fillna(0, subset=['arrivals', 'peak_arrivals'])
    .withColumn('net_flow', F.col('arrivals') - F.col('departures'))
    # Cast para timestamp agora que é string
    .withColumn('window_15min', F.expr("TIMESTAMP '1970-01-01 00:00:00' + window_15min"))
    .orderBy('station_id', 'window_15min')
)

df_demand_15min.printSchema()

root
 |-- station_id: string (nullable = true)
 |-- window_15min: timestamp (nullable = true)
 |-- station_name: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_peak_hour: boolean (nullable = false)
 |-- departures: long (nullable = false)
 |-- peak_departures: long (nullable = true)
 |-- avg_trip_duration_min: double (nullable = true)
 |-- arrivals: long (nullable = true)
 |-- peak_arrivals: long (nullable = true)
 |-- net_flow: long (nullable = true)



In [12]:
# ============================================================
# 3.4 Salvar demand_15min como Delta Lake
# ============================================================

path_15 = str(GOLD_DEMAND_15MIN).replace('\\', '/')

(
    df_demand_15min
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('day_of_week')
    .save(path_15)
)

DeltaTable.forPath(spark, path_15) \
    .history(1).select('version', 'timestamp', 'operation', 'operationMetrics') \
    .show(truncate=False)

df_demand_15min.createOrReplaceTempView('demand_15min')
print(f'demand_15min salvo → {GOLD_DEMAND_15MIN}')

+-------+-----------------------+---------+--------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                    |
+-------+-----------------------+---------+--------------------------------------------------------------------+
|0      |2026-04-24 13:45:49.595|WRITE    |{numFiles -> 77, numOutputRows -> 352916, numOutputBytes -> 2887843}|
+-------+-----------------------+---------+--------------------------------------------------------------------+

demand_15min salvo → /home/jovyan/work/dados/gold/demand_15min


---
## 4. Gold: `demand_30min`

Mesma lógica de `demand_15min` com janelas de 30 minutos.
Granularidade maior — usada para previsão de demanda de médio prazo.

In [13]:
# ============================================================
# 4.1 Partidas por estação × janela 30min
# ============================================================

df_dep_30 = (
    df_enriched
    .groupBy(
        F.col('start_station_id').alias('station_id'),
        F.col('start_station_name').alias('station_name'),
        'window_30min',
        'hour_of_day',
        'day_of_week',
        'is_peak_hour',
    )
    .agg(
        F.count('ride_id').alias('departures'),
        F.sum(
            F.when(F.col('is_peak_hour'), 1).otherwise(0)
        ).alias('peak_departures'),
        F.round(
            F.avg('trip_duration_sec') / 60.0, 2
        ).alias('avg_trip_duration_min'),
    )
)

df_arr_30 = (
    df_enriched
    .groupBy(
        F.col('end_station_id').alias('station_id'),
        'window_30min',
    )
    .agg(
        F.count('ride_id').alias('arrivals'),
        F.sum(
            F.when(F.col('is_peak_hour'), 1).otherwise(0)
        ).alias('peak_arrivals'),
    )
)

df_demand_30min = (
    df_dep_30
    .join(df_arr_30, on=['station_id', 'window_30min'], how='left')
    .fillna(0, subset=['arrivals', 'peak_arrivals'])
    .withColumn('net_flow', F.col('arrivals') - F.col('departures'))
    # Cast para timestamp agora que é string
    .withColumn('window_30min', F.expr("TIMESTAMP '1970-01-01 00:00:00' + window_30min"))
    .orderBy('station_id', 'window_30min')
)

df_demand_30min.printSchema()
print(f'demand_30min: {df_demand_30min.count():,} registros')
df_demand_30min.show(5, truncate=False)

root
 |-- station_id: string (nullable = true)
 |-- window_30min: timestamp (nullable = true)
 |-- station_name: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_peak_hour: boolean (nullable = false)
 |-- departures: long (nullable = false)
 |-- peak_departures: long (nullable = true)
 |-- avg_trip_duration_min: double (nullable = true)
 |-- arrivals: long (nullable = true)
 |-- peak_arrivals: long (nullable = true)
 |-- net_flow: long (nullable = true)

demand_30min: 287,086 registros
+----------+-------------------+-------------------+-----------+-----------+------------+----------+---------------+---------------------+--------+-------------+--------+
|station_id|window_30min       |station_name       |hour_of_day|day_of_week|is_peak_hour|departures|peak_departures|avg_trip_duration_min|arrivals|peak_arrivals|net_flow|
+----------+-------------------+-------------------+-----------+-----------+------------+------

In [14]:
# ============================================================
# 4.2 Salvar demand_30min como Delta Lake
# ============================================================

path_30 = str(GOLD_DEMAND_30MIN).replace('\\', '/')

(
    df_demand_30min
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('day_of_week')
    .save(path_30)
)

DeltaTable.forPath(spark, path_30) \
    .history(1).select('version', 'timestamp', 'operation') \
    .show(truncate=False)

df_demand_30min.createOrReplaceTempView('demand_30min')
print(f'demand_30min salvo  →  {GOLD_DEMAND_30MIN}')

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-04-24 13:45:59.966|WRITE    |
+-------+-----------------------+---------+

demand_30min salvo  →  /home/jovyan/work/dados/gold/demand_30min


---
## 5. Gold: `demand_rolling_avg` — Média Móvel + Alertas

**Entrada:** `demand_15min` (já calculado acima)

Calcula a **média móvel das últimas 4 janelas de 15 min (= 1 hora)**
e classifica cada ponto com `rebalance_alert`.

| Alert | Critério |
|---|---|
| `CRITICAL` | `net_flow < -5` **e** horário de pico |
| `WARNING` | `net_flow < -3` |
| `OK` | demais casos |

In [15]:
# ============================================================
# 5.1 Definir window spec para média móvel
# ============================================================

WINDOW_SIZE = 4   # 1 hora de histórico (4x15)

w_rolling = (
    Window
    .partitionBy('station_id')
    .orderBy('window_15min')
    .rowsBetween(-WINDOW_SIZE, -1)   # N janelas anteriores, menos a atual
)

df_rolling = (
    df_demand_15min
    # Média móvel de saídas e net_flow
    .withColumn(
        'rolling_avg_departures',
        F.round(F.avg('departures').over(w_rolling), 2),
    )
    .withColumn(
        'rolling_avg_net_flow',
        F.round(F.avg('net_flow').over(w_rolling), 2),
    )
    # Tendência: desvio da demanda atual vs média móvel
    .withColumn(
        'demand_trend',
        F.round(
            F.col('departures') - F.col('rolling_avg_departures'), 2
        ),
    )
    # Classificação de alerta de rebalanceamento
    .withColumn(
        'rebalance_alert',
        F.when(
            (F.col('net_flow') < -5) & F.col('is_peak_hour'),
            'CRITICAL',
        )
        .when(F.col('net_flow') < -3, 'WARNING')
        .otherwise('OK'),
    )
)

print(f'demand_rolling_avg: {df_rolling.count():,} registros')
print('\nDistribuição de alertas:')
df_rolling.groupBy('rebalance_alert').count().orderBy('rebalance_alert').show()

demand_rolling_avg: 352,916 registros

Distribuição de alertas:
+---------------+------+
|rebalance_alert| count|
+---------------+------+
|       CRITICAL|    82|
|             OK|345460|
|        WARNING|  7374|
+---------------+------+



In [16]:
# ============================================================
# 5.2 Visualizar alertas CRITICAL e WARNING
# ============================================================

print('🚨 Alertas CRITICAL (top 10 — estações em risco de esvaziamento em pico):')
(
    df_rolling
    .filter("rebalance_alert = 'CRITICAL'")
    .select(
        'station_id', 'station_name', 'window_15min',
        'departures', 'arrivals', 'net_flow',
        'rolling_avg_net_flow', 'demand_trend'
    )
    .orderBy(F.asc('net_flow'))
    .show(10, truncate=False)
)

print('Alertas WARNING (top 10):')
(
    df_rolling
    .filter("rebalance_alert = 'WARNING'")
    .select(
        'station_id', 'station_name', 'window_15min',
        'departures', 'arrivals', 'net_flow'
    )
    .orderBy(F.asc('net_flow'))
    .show(10, truncate=False)
)

🚨 Alertas CRITICAL (top 10 — estações em risco de esvaziamento em pico):
+----------+--------------------------+-------------------+----------+--------+--------+--------------------+------------+
|station_id|station_name              |window_15min       |departures|arrivals|net_flow|rolling_avg_net_flow|demand_trend|
+----------+--------------------------+-------------------+----------+--------+--------+--------------------+------------+
|6140.05   |W 21 St & 6 Ave           |2026-01-11 13:00:00|11        |0       |-11     |-4.5                |6.5         |
|5763.03   |Carmine St & 6 Ave        |2026-01-11 12:30:00|9         |0       |-9      |-2.0                |7.0         |
|6022.04   |E 16 St & 5 Ave           |2026-01-07 14:45:00|9         |0       |-9      |-2.0                |7.0         |
|6022.04   |E 16 St & 5 Ave           |2026-01-09 14:45:00|9         |0       |-9      |-1.25               |7.75        |
|6140.05   |W 21 St & 6 Ave           |2026-01-11 13:45:00|9      

In [17]:
# ============================================================
# 5.3 Salvar demand_rolling_avg como Delta Lake
# ============================================================

path_rolling = str(GOLD_DEMAND_ROLLING).replace('\\', '/')

(
    df_rolling
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .partitionBy('day_of_week')
    .save(path_rolling)
)

DeltaTable.forPath(spark, path_rolling) \
    .history(1).select('version', 'timestamp', 'operation') \
    .show(truncate=False)

df_rolling.createOrReplaceTempView('demand_rolling_avg')
print(f'demand_rolling_avg salvo  →  {GOLD_DEMAND_ROLLING}')

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-04-24 13:46:14.684|WRITE    |
+-------+-----------------------+---------+

demand_rolling_avg salvo  →  /home/jovyan/work/dados/gold/demand_rolling_avg


---
## 6. Gold: `station_hourly_profile`

**Grão:** `(start_station_id, hour_of_day, day_of_week)`

Perfil histórico médio de cada estação por hora e dia da semana.
Feature principal para o **Integrante 5** (feature engineering do modelo preditivo).

In [18]:
# ============================================================
# 6.1 Calcular perfil histórico por estação × hora × dia
# ============================================================

df_profile = (
    df_enriched
    .groupBy('start_station_id', 'hour_of_day', 'day_of_week')
    .agg(
        F.round(F.avg('trip_duration_sec') / 60.0, 2).alias('avg_duration_min'),
        F.count('ride_id').alias('total_trips'),
        F.countDistinct(F.to_date('started_at')).alias('days_observed'),
        F.round(
            F.sum(F.when(F.col('member_casual') == 'member', 1).otherwise(0)) /
            F.count('ride_id') * 100, 1
        ).alias('pct_member'),
        F.round(
            F.sum(F.when(F.col('rideable_type') == 'electric_bike', 1).otherwise(0)) /
            F.count('ride_id') * 100, 1
        ).alias('pct_electric'),
    )
    .withColumn(
        'avg_trips_per_day',
        F.round(
            F.col('total_trips') / F.greatest(F.col('days_observed'), F.lit(1)),
            2,
        ),
    )
    .orderBy('start_station_id', 'day_of_week', 'hour_of_day')
)

print(f'station_hourly_profile: {df_profile.count():,} perfis')
df_profile.show(5, truncate=False)

station_hourly_profile: 154,884 perfis
+----------------+-----------+-----------+----------------+-----------+-------------+----------+------------+-----------------+
|start_station_id|hour_of_day|day_of_week|avg_duration_min|total_trips|days_observed|pct_member|pct_electric|avg_trips_per_day|
+----------------+-----------+-----------+----------------+-----------+-------------+----------+------------+-----------------+
|1234.56         |2          |1          |18.05           |4          |2            |100.0     |100.0       |2.0              |
|1234.56         |14         |1          |9.77            |1          |1            |100.0     |100.0       |1.0              |
|1234.56         |18         |1          |6.53            |2          |1            |100.0     |100.0       |2.0              |
|1234.56         |1          |2          |29.85           |1          |1            |100.0     |100.0       |1.0              |
|1234.56         |17         |2          |5.01            |3     

In [19]:
# ============================================================
# 6.2 Salvar station_hourly_profile como Delta Lake
# ============================================================

path_profile = str(GOLD_STATION_PROFILE).replace('\\', '/')

(
    df_profile
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    # Sem partição
    .save(path_profile)
)

DeltaTable.forPath(spark, path_profile) \
    .history(1).select('version', 'timestamp', 'operation') \
    .show(truncate=False)

df_profile.createOrReplaceTempView('station_hourly_profile')
print(f'station_hourly_profile salvo  →  {GOLD_STATION_PROFILE}')

+-------+-----------------------+---------+
|version|timestamp              |operation|
+-------+-----------------------+---------+
|0      |2026-04-24 13:46:20.885|WRITE    |
+-------+-----------------------+---------+

station_hourly_profile salvo  →  /home/jovyan/work/dados/gold/station_hourly_profile


---
## 7. Análises SQL — Showcase do Pipeline

In [20]:
# ============================================================
# 7.1 Top 10 estações com maior pressão de saída em horário de pico
# ============================================================

print('=== TOP 10 ESTAÇÕES — MAIOR PRESSÃO DE SAÍDA EM PICO ===')
spark.sql("""
    SELECT
        station_id,
        station_name,
        COUNT(*) AS janelas_analisadas,
        SUM(departures) AS total_saidas,
        SUM(arrivals)   AS total_chegadas,
        SUM(net_flow)   AS net_flow_total,
        ROUND(AVG(departures), 1) AS media_saidas_por_janela,
        SUM(CASE WHEN rebalance_alert = 'CRITICAL' THEN 1 ELSE 0 END) AS alertas_criticos,
        SUM(CASE WHEN rebalance_alert = 'WARNING'  THEN 1 ELSE 0 END) AS alertas_warning
    FROM demand_rolling_avg
    WHERE is_peak_hour = true
    GROUP BY station_id, station_name
    ORDER BY alertas_criticos DESC, net_flow_total ASC
    LIMIT 10
""").show(truncate=False)

=== TOP 10 ESTAÇÕES — MAIOR PRESSÃO DE SAÍDA EM PICO ===
+----------+-------------------------+------------------+------------+--------------+--------------+-----------------------+----------------+---------------+
|station_id|station_name             |janelas_analisadas|total_saidas|total_chegadas|net_flow_total|media_saidas_por_janela|alertas_criticos|alertas_warning|
+----------+-------------------------+------------------+------------+--------------+--------------+-----------------------+----------------+---------------+
|6140.05   |W 21 St & 6 Ave          |174               |451         |6             |-445          |2.6                    |12              |27             |
|6022.04   |E 16 St & 5 Ave          |108               |195         |2             |-193          |1.8                    |6               |3              |
|6233.04   |Pier 61 At Chelsea Piers |147               |310         |10            |-300          |2.1                    |4               |16          

In [21]:
# ============================================================
# 7.2 Comparativo 15min vs 30min — mesmo horário de pico
# ============================================================

print('=== COMPARATIVO 15min vs 30min — PICO DA MANHÃ (8h) ===')
spark.sql("""
    SELECT
        d15.station_name,
        d15.window_15min,
        d15.departures   AS dep_15min,
        d15.net_flow     AS flow_15min,
        d30.departures   AS dep_30min,
        d30.net_flow     AS flow_30min
    FROM demand_15min d15
    JOIN demand_30min d30
        ON  d15.station_id = d30.station_id
        AND date_trunc('hour', d15.window_15min) = date_trunc('hour', d30.window_30min)
    WHERE d15.hour_of_day = 8
    ORDER BY dep_15min DESC
    LIMIT 15
""").show(truncate=False)

=== COMPARATIVO 15min vs 30min — PICO DA MANHÃ (8h) ===
+-------------------+-------------------+---------+----------+---------+----------+
|station_name       |window_15min       |dep_15min|flow_15min|dep_30min|flow_30min|
+-------------------+-------------------+---------+----------+---------+----------+
|W 21 St & 6 Ave    |2026-01-11 13:00:00|11       |-11       |14       |-14       |
|W 21 St & 6 Ave    |2026-01-11 13:00:00|11       |-11       |16       |-16       |
|W 21 St & 6 Ave    |2026-01-11 13:45:00|9        |-9        |14       |-14       |
|W 21 St & 6 Ave    |2026-01-11 13:45:00|9        |-9        |16       |-16       |
|5 Ave & E 63 St    |2026-01-08 13:45:00|9        |-8        |9        |-8        |
|5 Ave & E 63 St    |2026-01-08 13:45:00|9        |-8        |1        |1         |
|6 Ave & W 34 St    |2026-01-09 13:30:00|8        |-7        |2        |2         |
|W 21 St & 6 Ave    |2026-01-14 13:30:00|8        |-8        |11       |-11       |
|6 Ave & W 34 St    

In [22]:
# ============================================================
# 7.3 Perfil histórico — média de viagens por hora (Int. 5)
# ============================================================

print('=== PERFIL HISTÓRICO — TOP 5 ESTAÇÕES POR VOLUME MÉDIO/DIA ===')
spark.sql("""
    SELECT
        start_station_id,
        ROUND(SUM(avg_trips_per_day), 1) AS viagens_dia_total,
        ROUND(AVG(avg_duration_min), 1)  AS duracao_media_min,
        ROUND(AVG(pct_member), 1)        AS pct_member_medio,
        ROUND(AVG(pct_electric), 1)      AS pct_electric_medio
    FROM station_hourly_profile
    GROUP BY start_station_id
    ORDER BY viagens_dia_total DESC
    LIMIT 5
""").show(truncate=False)

=== PERFIL HISTÓRICO — TOP 5 ESTAÇÕES POR VOLUME MÉDIO/DIA ===
+----------------+-----------------+-----------------+----------------+------------------+
|start_station_id|viagens_dia_total|duracao_media_min|pct_member_medio|pct_electric_medio|
+----------------+-----------------+-----------------+----------------+------------------+
|6140.05         |1035.0           |9.7              |93.2            |69.9              |
|6233.04         |896.5            |11.5             |92.2            |72.5              |
|6492.08         |874.0            |8.5              |92.5            |74.7              |
|6602.03         |842.0            |8.9              |89.1            |73.1              |
|6331.01         |830.0            |9.4              |86.2            |80.7              |
+----------------+-----------------+-----------------+----------------+------------------+



---
## 8. Validação & Inventário Gold

In [23]:
# ============================================================
# 8.1 Inventário das tabelas Gold produzidas
# ============================================================

gold_tables = {
    'demand_15min':           str(GOLD_DEMAND_15MIN).replace('\\', '/'),
    'demand_30min':           str(GOLD_DEMAND_30MIN).replace('\\', '/'),
    'demand_rolling_avg':     str(GOLD_DEMAND_ROLLING).replace('\\', '/'),
    'station_hourly_profile': str(GOLD_STATION_PROFILE).replace('\\', '/'),
}

print('=' * 70)
print('INVENTÁRIO — GOLD LAYER (Integrante 4)')
print('=' * 70)
print(f'{"Tabela":<28} {"Registros":>12}  {"Versão":>7}  Partição')
print('-' * 70)

for name, path in gold_tables.items():
    try:
        cnt = spark.read.format('delta').load(path).count()
        ver = DeltaTable.forPath(spark, path).history(1).collect()[0]['version']
        part = 'day_of_week' if 'profile' not in name else '—'
        print(f'{name:<28} {cnt:>12,}  {ver:>7}  {part}')
    except Exception as e:
        print(f'{name:<28}  ERRO: {e}')

print('=' * 70)

INVENTÁRIO — GOLD LAYER (Integrante 4)
Tabela                          Registros   Versão  Partição
----------------------------------------------------------------------
demand_15min                      352,916        0  day_of_week
demand_30min                      287,086        0  day_of_week
demand_rolling_avg                352,916        0  day_of_week
station_hourly_profile            154,884        0  —


In [24]:
# ============================================================
# 8.2 Checklist de entrega
# ============================================================

checks = {
    'Silver/trips lida (Delta Lake)':             any(SILVER_TRIPS.rglob('*.parquet')),
    'Janelas 15min calculadas':                   any(GOLD_DEMAND_15MIN.rglob('*.parquet')),
    'Janelas 30min calculadas':                   any(GOLD_DEMAND_30MIN.rglob('*.parquet')),
    'Média móvel + rebalance_alert':              any(GOLD_DEMAND_ROLLING.rglob('*.parquet')),
    'Perfil histórico estação × hora':            any(GOLD_STATION_PROFILE.rglob('*.parquet')),
    'Escrita em Delta Lake':                      True,
    'Particionamento por day_of_week':            True,
    'Sem UDFs (transformações colunares puras)':  True,
}

print('\n' + '=' * 60)
print('CHECKLIST — INTEGRANTE 4')
print('=' * 60)
for item, status in checks.items():
    print(f'  {"OK" if status else "Falha"}  {item}')

total_ok = sum(checks.values())
print(f'\n  Progresso: {total_ok}/{len(checks)} ({100*total_ok//len(checks)}%)')
print('=' * 60)


CHECKLIST — INTEGRANTE 4
  OK  Silver/trips lida (Delta Lake)
  OK  Janelas 15min calculadas
  OK  Janelas 30min calculadas
  OK  Média móvel + rebalance_alert
  OK  Perfil histórico estação × hora
  OK  Escrita em Delta Lake
  OK  Particionamento por day_of_week
  OK  Sem UDFs (transformações colunares puras)

  Progresso: 8/8 (100%)


In [25]:
# ============================================================
# 8.3 Liberar cache e encerrar
# ============================================================

df_enriched.unpersist()
print('Cache liberado.')

# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada!")

Cache liberado.
SparkSession encerrada!
